In [1]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

In [21]:
RUTA_CSV_EQUIPO = '/content/drive/MyDrive/_SE CIENCIA DE DATOS/_Programacion para Ciencia de Datos/Sesion 12 - Modelo predictivo - Clasificacion/df_limpio_procesado.pkl'
df = pd.read_pickle(RUTA_CSV_EQUIPO )
df.head()

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration,module_presentation_length,total_assessments_submitted,average_score,total_weighted_score,total_clicks,unique_days_active
0,aaa,2013j,11391,m,east anglian region,he qualification,90-100%,mayor o igual a 55 años,0,240,n,pass,-159,268,268,5,82.0,82.4,934,40
1,aaa,2013j,28400,f,scotland,he qualification,20-30%,entre 35 y 55 años,0,60,n,pass,-53,268,268,5,66.4,65.4,1435,80
2,aaa,2013j,30268,f,north western region,a level or equivalent,30-40%,entre 35 y 55 años,0,60,y,withdrawn,-92,12,268,0,0.0,0.0,281,12
3,aaa,2013j,31604,f,south east region,a level or equivalent,50-60%,entre 35 y 55 años,0,60,n,pass,-52,268,268,5,76.0,76.3,2158,123
4,aaa,2013j,32885,f,west midlands region,lower than a level,50-60%,menor o igual a 35 años,0,60,n,pass,-176,268,268,5,54.4,55.0,1034,70


In [22]:
# Verificar la cantidad de registros por clase
df['final_result'].value_counts()
# Debido a que las clases estan desbalancedas, aplicaremos submuestreo para que las clases mayritarias en frecuencia no aprendan mas que otras
# Vamos a reducir las clases grandes hasta que todas tengan el tamaño de la clase más pequeña (3 024).
# A este proceso se llama Submuestreo de las clases mayoritarias (Random Undersampling).

,count
final_result,
pass,12361
withdrawn,10156
fail,7052
distinction,3024


In [ ]:
# Debido al desbalance existente en la variable objetivo final_result, se aplicó la técnica de submuestreo aleatorio de las clases mayoritarias
# (Random Undersampling). Esta técnica consiste en reducir el número de registros de las clases con mayor cantidad de observaciones hasta igualarlas
# con la clase minoritaria, obteniendo un conjunto de datos balanceado que favorece un entrenamiento más equilibrado del modelo de clasificación.

In [23]:
# Aplicando submuestreo
from sklearn.utils import resample

# Separar cada clase
pass_df = df[df['final_result'] == 'pass']
withdrawn_df = df[df['final_result'] == 'withdrawn']
fail_df = df[df['final_result'] == 'fail']
distinction_df = df[df['final_result'] == 'distinction']

# Tamaño de la clase minoritaria
n = len(distinction_df)

# Submuestreo de las clases mayoritarias
pass_down = resample(pass_df, replace=False, n_samples=n, random_state=42)

withdrawn_down = resample(withdrawn_df, replace=False, n_samples=n,random_state=42)

fail_down = resample(fail_df, replace=False, n_samples=n, random_state=42)

# Unir todas las clases
df_balanceado = pd.concat([pass_down, withdrawn_down, fail_down, distinction_df])

# Mezclar aleatoriamente el conjunto de datos
df_balanceado = df_balanceado.sample(frac=1, random_state=42).reset_index(drop=True)

In [24]:
df_balanceado['final_result'].value_counts()

,count
final_result,
distinction,3024
fail,3024
pass,3024
withdrawn,3024


In [ ]:
# Total de registros 3024×4=12096

In [25]:
# Eliminado columnas que no tiene aporte
df = df_balanceado.drop(columns=[
    'code_module',
    'code_presentation',
    'id_student'
])
df.head()

,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration,module_presentation_length,total_assessments_submitted,average_score,total_weighted_score,total_clicks,unique_days_active
0,f,london region,he qualification,10-20%,menor o igual a 35 años,0,60,n,pass,-68,240,240,12,84.083333,83.25,3067,81
1,f,london region,he qualification,90-100%,entre 35 y 55 años,0,60,n,distinction,-60,269,269,12,93.000000,92.50,13701,266
2,f,south west region,a level or equivalent,80-90%,menor o igual a 35 años,0,90,y,pass,-50,262,262,7,81.000000,167.30,847,74
3,m,south west region,no formal quals,30-40%,menor o igual a 35 años,0,120,n,withdrawn,-52,11,268,0,0.000000,0.00,0,0
4,f,south region,a level or equivalent,80-90%,menor o igual a 35 años,0,60,y,fail,-60,234,234,8,84.125000,57.53,187,26


In [26]:
# Separar X y Y
X = df.drop(columns='final_result')
y = df['final_result']

In [27]:
# Visualizar los dominios de cada variables categorica
# Lista de variables categóricas
variables_categoricas = [
    'gender',
    'region',
    'highest_education',
    'imd_band',
    'age_band',
    'disability',
    'final_result'
]

# Mostrar valores únicos de cada variable
for variable in variables_categoricas:
    print("="*50)
    print("Variable:", variable)
    print("Cantidad de categorías:", df[variable].nunique())
    print(df[variable].unique())
    print()

Variable: gender
Cantidad de categorías: 2
['f', 'm']
Categories (2, object): ['f', 'm']

Variable: region
Cantidad de categorías: 13
['london region', 'south west region', 'south region', 'east midlands region', 'east anglian region', ..., 'west midlands region', 'south east region', 'scotland', 'ireland', 'north region']
Length: 13
Categories (13, object): ['east anglian region', 'east midlands region', 'ireland', 'london region', ...,
                          'south west region', 'wales', 'west midlands region', 'yorkshire region']

Variable: highest_education
Cantidad de categorías: 5
['he qualification', 'a level or equivalent', 'no formal quals', 'lower than a level', 'post graduate qualification']
Categories (5, object): ['no formal quals' < 'lower than a level' < 'a level or equivalent' <
                         'he qualification' < 'post graduate qualification']

Variable: imd_band
Cantidad de categorías: 10
['10-20%', '90-100%', '80-90%', '30-40%', '20-30%', '70-80%', '50-6

In [28]:
# Codificando
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Crear copia del dataset original
df_codificado = df.copy()

# 1. VARIABLES ORDINALES
# age_band
df_codificado['age_band'] = df_codificado['age_band'].map({
    'menor o igual a 35 años': 1,
    'entre 35 y 55 años': 2,
    'mayor o igual a 55 años': 3
})

# highest_education
df_codificado['highest_education'] = df_codificado['highest_education'].map({
    'no formal quals': 1,
    'lower than a level': 2,
    'a level or equivalent': 3,
    'he qualification': 4,
    'post graduate qualification': 5
})

# imd_band
df_codificado['imd_band'] = df_codificado['imd_band'].map({
    '0-10%': 1,
    '10-20%': 2,
    '20-30%': 3,
    '30-40%': 4,
    '40-50%': 5,
    '50-60%': 6,
    '60-70%': 7,
    '70-80%': 8,
    '80-90%': 9,
    '90-100%': 10
})

# 2. VARIABLE BINARIA
df_codificado['disability'] = df_codificado['disability'].map({
    'n': 0,
    'y': 1
})

# 3. VARIABLES NOMINALES
# One Hot Encoding
# drop_first=True evita multicolinealidad
# dtype=int convierte True/False a 1/0

df_codificado = pd.get_dummies(
    df_codificado,
    columns=[
        'gender',
        'region'
    ],
    drop_first=True,
    dtype=int
)

# 4. VARIABLE OBJETIVO
label_encoder = LabelEncoder()
df_codificado['final_result'] = label_encoder.fit_transform(
    df_codificado['final_result']
)

df_codificado.head()

,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,date_unregistration,module_presentation_length,...,region_london region,region_north region,region_north western region,region_scotland,region_south east region,region_south region,region_south west region,region_wales,region_west midlands region,region_yorkshire region
0,4,2,1,0,60,0,2,-68,240,240,...,1,0,0,0,0,0,0,0,0,0
1,4,10,2,0,60,0,0,-60,269,269,...,1,0,0,0,0,0,0,0,0,0
2,3,9,1,0,90,1,2,-50,262,262,...,0,0,0,0,0,0,1,0,0,0
3,1,4,1,0,120,0,3,-52,11,268,...,0,0,0,0,0,0,1,0,0,0
4,3,9,1,0,60,1,1,-60,234,234,...,0,0,0,0,0,1,0,0,0,0


In [29]:
# Eliminar la variable date_unregistration
df_codificado = df_codificado.drop(columns=['date_unregistration'])
df_codificado.head()

,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,date_registration,module_presentation_length,total_assessments_submitted,...,region_london region,region_north region,region_north western region,region_scotland,region_south east region,region_south region,region_south west region,region_wales,region_west midlands region,region_yorkshire region
0,4,2,1,0,60,0,2,-68,240,12,...,1,0,0,0,0,0,0,0,0,0
1,4,10,2,0,60,0,0,-60,269,12,...,1,0,0,0,0,0,0,0,0,0
2,3,9,1,0,90,1,2,-50,262,7,...,0,0,0,0,0,0,1,0,0,0
3,1,4,1,0,120,0,3,-52,268,0,...,0,0,0,0,0,0,1,0,0,0
4,3,9,1,0,60,1,1,-60,234,8,...,0,0,0,0,0,1,0,0,0,0


In [ ]:
# "La variable date_unregistration presentó valores nulos que representaban estudiantes que no cancelaron su matrícula. Inicialmente para conservar esta información semántica,
# los valores nulos de estudiantes que finalizaron el módulo fueron reemplazados por la duración total del módulo, indicando permanencia hasta la finalización."

# EXCLUSION DE LA VARIABLE date_unregistration
# Finalmente la variable date_unregistration fue excluida del conjunto de variables predictoras debido a que representa un evento posterior dentro de la trayectoria académica
# del estudiante y está directamente asociada con la categoría final withdrawn de la variable objetivo final_result. Su inclusión podría generar una dependencia
# excesiva del modelo hacia la condición de retiro, reduciendo la capacidad de identificar otros factores académicos y de interacción del estudiante que influyen
# en el resultado final."

**La técnica feature_importances_ de Random Forest**

Permite identificar la contribución de cada variable independiente en la predicción de la variable objetivo. Para ello, primero se entrena un bosque de árboles de decisión mediante el método fit(), y posteriormente se calcula un puntaje de importancia para cada variable según su aporte al proceso de clasificación.

En este estudio, esta técnica se utilizó para seleccionar las variables más influyentes en la predicción del resultado final del estudiante (final_result), las cuales fueron empleadas en la construcción del modelo de regresión logística.

In [31]:
# Selección de features con RF feature_importances_ → top-9
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

# Separar variables predictoras y objetivo

X = df_codificado.drop('final_result', axis=1)
y = df_codificado['final_result']


# Lista de nombres de variables predictoras
features_names = X.columns.tolist()

# Convertir a matriz numpy
X_raw = X.values
y_raw = y.values

# Entrenar Random Forest
rf_sel = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf_sel.fit(X_raw, y_raw)

# Obtener importancia de variables
importancias = (
    pd.Series(
        rf_sel.feature_importances_,
        index=features_names
    )
    .sort_values(ascending=False)
)

# Seleccionar las 9 variables más importantes
top_features = importancias.head(9).index.tolist()

print("Importancia de todas las variables:")
for i, (variable, importancia) in enumerate(importancias.items(), 1):
    print(f"{i:>2}. {variable:<60} ({importancia:.4f})")

Importancia de todas las variables:
 1. total_weighted_score                                         (0.1832)
 2. average_score                                                (0.1726)
 3. total_assessments_submitted                                  (0.1216)
 4. unique_days_active                                           (0.1118)
 5. total_clicks                                                 (0.0966)
 6. date_registration                                            (0.0668)
 7. imd_band                                                     (0.0413)
 8. studied_credits                                              (0.0360)
 9. module_presentation_length                                   (0.0352)
10. highest_education                                            (0.0209)
11. gender_m                                                     (0.0134)
12. age_band                                                     (0.0116)
13. num_of_prev_attempts                                         (0.0099)
14

**JUSTFIFICACION DE LA ELECCION DE LAS PRIMERAS 9 VARIABLES**

Se seleccionaron las nueve variables con mayor importancia porque, a partir de la décima posición, los valores de importancia disminuyen considerablemente, indicando una menor contribución al proceso de clasificación. Con esta selección se conservan las variables con mayor capacidad predictiva y se reduce la complejidad del modelo al eliminar variables cuyo aporte es marginal.

Si sumamos las importancias de las primeras nueve variables: 0.2666 + 0.1705 + 0.1529 + 0.1118 + 0.0663 + 0.0615 + 0.0349 + 0.0229 + 0.0199 ≈ 0.9073

Es decir, las nueve primeras variables concentran aproximadamente el 90.7% de la importancia total.

Las variables categóricas fueron codificadas mediante One-Hot Encoding, por lo que algunas variables, como region, se dividieron en varias variables binarias. En consecuencia, su importancia quedó distribuida entre dichas columnas, mientras que las variables numéricas conservaron un único valor de importancia.

Interpretación de las variables seleccionadas

1. total_weighted_score (0.1832): Es la variable más influyente. Resume el desempeño acumulado del estudiante en las evaluaciones ponderadas del módulo.

2. average_score (0.1726): Muestra el rendimiento promedio obtenido en las actividades evaluativas.

3. total_assessments_submitted (0.1216): Representa compromiso académico: entregar evaluaciones está asociado con mejores resultados.

4. unique_days_active (0.1118): Mide constancia de interacción con la plataforma. Un estudiante activo durante más días tiene mayor probabilidad de obtener mejores resultados.

5. total_clicks (0.0966): Representa la intensidad de interacción con los recursos virtuales.

6. date_registration (0.0668): Ahora aparece como una variable temporal razonable. Puede indicar diferencias entre estudiantes que se matricularon anticipadamente o tardíamente.

7. imd_band (0.0413): Aporta información socioeconómica del estudiante.

8. studied_credits (0.0360): Representa la carga académica acumulada del estudiante.

9. module_presentation_length (0.0352): Representa la duración del módulo y puede capturar diferencias entre presentaciones del curso.

**Rendimiento académico:**
total_weighted_score
average_score
total_assessments_submitted

**Participación en la plataforma:**
unique_days_active
total_clicks

In [32]:
# Variables dependiente e independientes
X_final = df_codificado[top_features]
y_final = df_codificado['final_result']

In [33]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_final,
    y_final,
    test_size=0.2,  # 20% para el test y 80% para el entrenamiento
    random_state=42,
    stratify=y_final  # Le indica a train_test_split() que mantenga la misma proporción de clases de la variable objetivo (y_final) tanto en el conjunto de entrenamiento como en el de prueba.
)

In [ ]:
# Total de filas:  4x3024 = 12 096 registros
# Entrenamiento 80% = 9676
# Prueba 20% = 2420

**Escalamiento de las variables predictoras**

MEediante StandardScaler antes de entrenar la regresión logística, debido a que este algoritmo es sensible a la escala de las variables. El escalamiento permite que todas las variables tengan una magnitud comparable, favoreciendo una estimación más estable de los coeficientes y una mejor convergencia del proceso de entrenamiento.

In [34]:
# Escalar
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

y_train contiene 4 clases

- 0  -> distinction
- 1  -> fail
- 2  -> pass
- 3  -> withdrawn

Al ejecutar clf.fit(X_train_scaled, y_train). La regresión logística detecta que existen 4 clases distintas, por lo que entrena automáticamente un modelo multiclase.

In [35]:
# Aplicando regresion logistica
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(
    max_iter=500,
    random_state=42
)
clf.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=500, random_state=42)

In [36]:
y_pred = clf.predict(X_test_scaled)
print(f'Accuracy: {(y_pred == y_test).mean() * 100:.2f} %')

Accuracy: 66.49 %


La exactitud (Accuracy) del modelo fue de 66.49%, lo que significa que el modelo clasificó correctamente el 66.49% de las observaciones del conjunto de prueba.



En el proyecto del grupo el total de registros en el conjunto de prueba: 2420.
Accuracy: 66.49%. Eso significa que el modelo clasificó correctamente aproximadamente: 2420×0.6649≈1609

Alrededor de 1609 estudiantes fueron clasificados correctamente, mientras que aproximadamente 811 fueron clasificados en una categoría distinta a la real.

In [37]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.74      0.83      0.78       605
           1       0.54      0.43      0.48       605
           2       0.66      0.63      0.65       605
           3       0.68      0.77      0.72       605

    accuracy                           0.66      2420
   macro avg       0.66      0.66      0.66      2420
weighted avg       0.66      0.66      0.66      2420



La funcion classification_report(y_test, y_pred) Recibe dos parámetros principales que son y_test, y_pred

- y_test → contiene las clases reales.
- y_pred → contiene las clases predichas por el modelo.

La función compara ambas listas y calcula automáticamente las métricas de evaluación de la clasificación.

Las 4 métricas más usadas para evaluar algoritmos de clasificación son:
| # | Inglés | Español | ¿Qué mide? |
|---|--------|---------|------------|
| 1 | Accuracy | Exactitud | Proporción total de predicciones correctas sobre el total |
| 2 | Precision | Precisión | De lo que predije como positivo, ¿cuánto era realmente positivo? |
| 3 | Recall (Sensitivity) | Sensibilidad / Exhaustividad | De todo lo que era realmente positivo, ¿cuánto detecté? |
| 4 | F1-Score | Puntuación F1 | Media armónica entre Precision y Recall (equilibrio entre ambas) |

| Clase           | Precisión (Precision) | Sensibilidad (Recall) | F1-score | Interpretación                                                                                                                                                                                                                                                                                                                                                                                                  |
| --------------- | --------------------: | --------------------: | -------: | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **distinction** |                  0.74 |                  0.83 |     0.78 | El modelo tiene un buen desempeño identificando estudiantes con `distinction`. La **precisión (Precision)** de 0.74 indica que, de todos los estudiantes que el modelo clasificó como `distinction`, el 74% realmente pertenece a esa categoría. La **sensibilidad (Recall)** de 0.83 indica que el modelo logra identificar correctamente al 83% de los estudiantes que realmente obtuvieron `distinction`.    |
| **fail**        |                  0.54 |                  0.43 |     0.48 | Es la categoría con menor desempeño. La **precisión (Precision)** de 0.54 indica que solo el 54% de los estudiantes predichos como `fail` realmente pertenecían a esta categoría. La **sensibilidad (Recall)** de 0.43 muestra que el modelo solo logra detectar al 43% de los estudiantes que realmente desaprobaron, por lo que una proporción importante de casos `fail` es confundida con otras categorías. |
| **pass**        |                  0.66 |                  0.63 |     0.65 | El modelo presenta un desempeño moderado. La **precisión (Precision)** de 0.66 indica que aproximadamente dos tercios de las predicciones como `pass` son correctas. La **sensibilidad (Recall)** de 0.63 muestra que identifica correctamente al 63% de los estudiantes que realmente aprobaron.                                                                                                               |
| **withdrawn**   |                  0.68 |                  0.77 |     0.72 | El modelo presenta una buena capacidad para identificar estudiantes retirados. La **precisión (Precision)** de 0.68 indica que el 68% de los estudiantes predichos como `withdrawn` realmente pertenecían a esa categoría. La **sensibilidad (Recall)** de 0.77 indica que el modelo logra detectar correctamente al 77% de los estudiantes que realmente abandonaron el módulo.                                |




In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=clf.classes_,
    columns=clf.classes_
)

cm_df

,0,1,2,3
0,502,28,75,0
1,18,260,105,222
2,143,81,381,0
3,13,113,13,466


| Real \ Predicho     | distinction (0) | fail (1) | pass (2) | withdrawn (3) |
| ------------------- | --------------: | -------: | -------: | ------------: |
| **distinction (0)** |             502 |       28 |       75 |             0 |
| **fail (1)**        |              18 |      260 |      105 |           222 |
| **pass (2)**        |             143 |       81 |      381 |             0 |
| **withdrawn (3)**   |              13 |      113 |       13 |           466 |


Total test: 2420. Existe 605 registros de cada clase para las pruebas

**1. distinction (clase 0)**
De los estudiantes que realmente obtuvieron distinction:

- 502 fueron correctamente clasificados como distinction.
- 28 fueron confundidos como fail.
- 75 fueron confundidos como pass.

El modelo reconoce bien esta categoría, aunque existe cierta confusión con estudiantes aprobados (pass). Esto coincide con su sensibilidad alta (0.83).


**2. fail (clase 1)**

De los estudiantes que realmente desaprobaron:

- 260 fueron correctamente identificados como fail.
- 18 fueron clasificados como distinction.
- 105 fueron clasificados como pass.
- 222 fueron clasificados como withdrawn.

Esta es la clase más problemática del modelo. Aunque identifica algunos casos correctamente, una cantidad importante de estudiantes que realmente desaprobaron son confundidos principalmente con withdrawn y pass. Esto explica el bajo valor de sensibilidad (0.43) obtenido para fail.


**3. pass (clase 2)**

De los estudiantes que realmente aprobaron:

- 381 fueron correctamente clasificados como pass.
- 143 fueron confundidos como distinction.
- 81 fueron confundidos como fail.

El modelo tiene un desempeño moderado para esta clase. La principal confusión ocurre entre estudiantes aprobados y estudiantes con distinction, debido a que ambos representan resultados académicos positivos y pueden compartir características similares.


**4. withdrawn (clase 3)**

De los estudiantes que realmente abandonaron:

- 466 fueron correctamente clasificados como withdrawn.
- 113 fueron confundidos como fail.
- 13 fueron confundidos como distinction.
- 13 fueron confundidos como pass.

El modelo identifica adecuadamente a los estudiantes retirados, aunque existe confusión con estudiantes que finalmente desaprobaron. Esto explica la alta sensibilidad (0.77) de esta clase.